# 🚢 鐵達尼號：AI 預測誰會活下來？

**故事背景：** 1912 年鐵達尼號沉沒，1,309 位乘客中約 38% 生還。

**任務：** 我們要訓練一個 AI 預測誰會活下來，並分析原因。

---

▶️ **執行方式：** 上方選單點 **「執行階段 → 全部執行」**（或按 `Ctrl + F9`），然後往下捲動看結果。

<sub>資料來源：OpenML Titanic（id 40945），原始整理者 Frank E. Harrell Jr. & Thomas Cason（Vanderbilt University），資料源自 Encyclopedia Titanica。</sub>

In [ ]:
#@title ⚙️ 環境準備（執行即可）
import io, os, sys, subprocess, warnings, urllib.request
warnings.filterwarnings("ignore")

# 1/3 載入套件
try:
    import shap, xgboost
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shap", "xgboost"])
    import shap, xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import display
from xgboost import XGBClassifier
from sklearn.datasets import fetch_openml
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
print("1/3 載入套件 ✅")

# 2/3 中文字型
FONT_PATH = "NotoSansTC.ttf"
try:
    if not os.path.exists(FONT_PATH):
        url = "https://github.com/google/fonts/raw/main/ofl/notosanstc/NotoSansTC%5Bwght%5D.ttf"
        with urllib.request.urlopen(url, timeout=60) as r, open(FONT_PATH, "wb") as f:
            f.write(r.read())
    font_manager.fontManager.addfont(FONT_PATH)
    plt.rcParams["font.family"] = font_manager.FontProperties(fname=FONT_PATH).get_name()
    print("2/3 中文字型 ✅")
except Exception:
    print("2/3 中文字型 ⚠️ 下載失敗，圖上的中文可能顯示成方塊（不影響結果）")
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

# 3/3 資料
# 資料來源：OpenML Titanic (id 40945)，原始整理者 Frank E. Harrell Jr. & Thomas Cason
# (Vanderbilt University)，資料源自 Encyclopedia Titanica。僅供教學使用。
DATA_URL = "https://raw.githubusercontent.com/PatrickforJob/titanic-shap-demo/main/titanic.csv"
try:
    with urllib.request.urlopen(DATA_URL, timeout=30) as r:
        df = pd.read_csv(io.BytesIO(r.read()))
except Exception:
    print("   GitHub 讀取失敗，改從 OpenML 下載（可能需要 1–2 分鐘）…")
    df = fetch_openml("titanic", version=1, as_frame=True).frame
df["survived"] = df["survived"].astype(str).astype(int)
df["pclass"] = df["pclass"].astype(int)
print("3/3 載入資料 ✅")

# 小工具
def train(model, X, y):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    model.fit(Xtr, ytr)
    return model, accuracy_score(yte, model.predict(Xte))

def new_xgb():
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                         subsample=0.9, random_state=42, eval_metric="logloss")

def explain(model, X):
    try:
        sv = shap.TreeExplainer(model)(X)
        if sv.values.ndim == 3:
            sv = sv[:, :, 1]
        return sv
    except Exception:
        # 備援：shap 與 xgboost 版本不相容時，改用 XGBoost 內建的 SHAP 計算
        import xgboost as xgb
        contrib = model.get_booster().predict(xgb.DMatrix(X), pred_contribs=True)
        return shap.Explanation(values=contrib[:, :-1], base_values=contrib[:, -1],
                                data=X.values, feature_names=list(X.columns))

def shap_beeswarm(sv, title):
    shap.plots.beeswarm(sv, max_display=len(sv.feature_names), show=False)
    fig = plt.gcf()
    ax = fig.axes[0]
    ax.set_title(title, fontsize=14, loc="left")
    ax.set_xlabel("SHAP 值（往右 = 推向生還，往左 = 推向罹難）")
    try:                                   # 右側色條改中文
        cb = fig.axes[-1]
        cb.set_ylabel("特徵數值", fontsize=11)
        cb.set_yticklabels(["低", "高"])
    except Exception:
        pass
    plt.tight_layout(); plt.show()

print(f"\n✅ 準備完成，共 {len(df)} 位乘客")

## 1. 認識資料：我們有哪些特徵？

In [ ]:
#@title ▶️ 資料長什麼樣子
print(f"乘客人數：{len(df)}　欄位數：{df.shape[1]}\n")
print("📋 資料範例（前 5 筆）")
display(df.head(5))

info = pd.DataFrame([
    ("pclass",    "艙等",               "1 = 頭等、2 = 二等、3 = 三等"),
    ("survived",  "是否生還",           "1 = 生還、0 = 罹難（標籤）"),
    ("name",      "姓名",               ""),
    ("sex",       "性別",               "female = 女、male = 男"),
    ("age",       "年齡",               "歲"),
    ("sibsp",     "同行兄弟姊妹/配偶數", "人"),
    ("parch",     "同行父母/子女數",     "人"),
    ("ticket",    "船票號碼",           ""),
    ("fare",      "票價",               "英鎊"),
    ("cabin",     "艙房號碼",           ""),
    ("embarked",  "登船港口",           "C = 瑟堡、Q = 皇后鎮、S = 南安普敦"),
    ("boat",      "救生艇編號",         ""),
    ("body",      "遺體編號",           ""),
    ("home.dest", "家鄉 / 目的地",      ""),
], columns=["欄位", "中文名稱", "說明"])
info["缺值數（比例）"] = [f"{df[c].isna().sum()}（{df[c].isna().mean():.1%}）" for c in info["欄位"]]
print("\n📋 特徵名稱與說明")
display(info)

## 2. 資料標籤：定義與分布

In [ ]:
#@title ▶️ 標籤定義與分布
label = pd.DataFrame({
    "survived": [0, 1],
    "意義": ["罹難", "生還"],
    "人數": [int((df["survived"] == 0).sum()), int((df["survived"] == 1).sum())],
})
label["比例"] = (label["人數"] / label["人數"].sum()).map("{:.1%}".format)
display(label)

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(label["意義"], label["人數"], color=["#8d99ae", "#2a9d8f"])
for b, n, p in zip(bars, label["人數"], label["比例"]):
    ax.text(b.get_x() + b.get_width() / 2, n, f"{n} 人\n{p}", ha="center", va="bottom")
ax.set_title("標籤分布", fontsize=14, loc="left")
ax.set_ylabel("人數"); ax.set_ylim(0, label["人數"].max() * 1.2)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 3. 資料前處理

In [ ]:
#@title ▶️ 前處理：補缺值、轉成數字
data = pd.DataFrame(index=df.index)
data["艙等"] = df["pclass"]
data["性別(女=1)"] = (df["sex"] == "female").astype(int)
data["年齡"] = df["age"].fillna(df["age"].median())
data["同行兄弟姊妹配偶數"] = df["sibsp"]
data["同行父母子女數"] = df["parch"]
data["票價"] = df["fare"].fillna(df["fare"].median())
emb = df["embarked"].fillna(df["embarked"].mode()[0])
data["登船_瑟堡"] = (emb == "C").astype(int)
data["登船_皇后鎮"] = (emb == "Q").astype(int)
data["登船_南安普敦"] = (emb == "S").astype(int)
data["有救生艇編號"] = df["boat"].notna().astype(int)
data["有遺體編號"] = df["body"].notna().astype(int)
y = df["survived"].astype(int)

BASE = ["艙等", "性別(女=1)", "年齡", "同行兄弟姊妹配偶數", "同行父母子女數", "票價",
        "登船_瑟堡", "登船_皇后鎮", "登船_南安普敦"]
LEAK = ["有救生艇編號", "有遺體編號"]

X = data[BASE]
print("✅ 前處理完成（細節略過）")

## 4. 決策樹

In [ ]:
#@title ▶️ 決策樹：訓練
tree, acc_tree = train(DecisionTreeClassifier(max_depth=3, random_state=42), X, y)
print(f"使用特徵：{len(X.columns)} 個")
print(f"準確率 Accuracy：{acc_tree:.1%}")

In [ ]:
#@title ▶️ 決策樹：樹狀圖
fig, ax = plt.subplots(figsize=(18, 8))
nodes = plot_tree(tree, feature_names=list(X.columns), class_names=["罹難", "生還"],
                  filled=True, rounded=True, impurity=False, proportion=True, fontsize=10, ax=ax)
for t in nodes:                          # 把英文標示改成中文
    s = t.get_text()
    s = s.replace("samples = ", "佔全部乘客 ").replace("value = ", "[罹難, 生還] = ").replace("class = ", "預測：")
    t.set_text(s)
ax.set_title("決策樹（左邊 = 條件成立 True，右邊 = 不成立 False）", fontsize=14, loc="left")
plt.tight_layout(); plt.show()

## 5. XGBoost

In [ ]:
#@title ▶️ XGBoost：訓練
xgb_model, acc_xgb = train(new_xgb(), X, y)
print(f"使用特徵：{len(X.columns)} 個")
print(f"準確率 Accuracy：決策樹 {acc_tree:.1%}　XGBoost {acc_xgb:.1%}")

## 6. SHAP：XGBoost 憑什麼判斷？

In [ ]:
#@title ▶️ XGBoost SHAP
sv = explain(xgb_model, X)
shap_beeswarm(sv, "XGBoost SHAP（每個點 = 一位乘客）")

## 7. 資料洩漏範例

In [ ]:
#@title ▶️ 加入「救生艇編號」「遺體編號」重新訓練 XGBoost
X_leak = data[BASE + LEAK]
leak_model, acc_leak = train(new_xgb(), X_leak, y)
print(f"加入特徵：{'、'.join(LEAK)}")
print(f"使用特徵：{len(X_leak.columns)} 個")
print(f"準確率 Accuracy：原本 {acc_xgb:.1%} ➜ 加入後 {acc_leak:.1%}")

In [ ]:
#@title ▶️ 資料洩漏範例 SHAP
sv_leak = explain(leak_model, X_leak)
shap_beeswarm(sv_leak, "加入洩漏特徵後的 SHAP（每個點 = 一位乘客）")